# HTR CREMMA Medieval — Fine-tuning TrOCR + Baseline
**Projet MD5 2026 — HETIC** | Axe : *Impact des abreviations medievales sur les performances HTR*

> **IMPORTANT** : ne relancez jamais une cellule deja executee (surtout le clone/cd).  
> Utilisez `Exécution > Tout exécuter` depuis le debut a chaque nouvelle session.

**Avant de lancer :** `Exécution > Modifier le type d'exécution > GPU (T4)`

## 0 — GPU

In [1]:
import torch
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('Pas de GPU — activez-le dans Exécution > Modifier le type d\'exécution')

GPU : Tesla T4
VRAM : 15.6 GB


## 1 — Clone + setup (executer UNE SEULE FOIS par session)

In [2]:
import os, sys

REPO_URL  = 'https://github.com/SebastianCaicedo17/Zekraoui_CREMMA-M-di-val_MD5.git'
REPO_NAME = 'repo_htr'
REPO_PATH = f'/kaggle/working/{REPO_NAME}'

if not os.path.exists(REPO_PATH):
    os.system(f'git clone --branch pretraitement_segmentation {REPO_URL} {REPO_PATH}')
else:
    os.system(f'git -C {REPO_PATH} pull')
    print('Repo deja present, pull effectue.')

os.chdir(REPO_PATH)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print('CWD    :', os.getcwd())
print('src/   :', os.listdir('src/'))

from src.utils import fixer_seeds
from src.htr   import construire_dataset
print('Imports OK')


Cloning into '/kaggle/working/repo_htr'...


CWD    : /kaggle/working/repo_htr
src/   : ['preprocessing.py', 'data_loader.py', '__init__.py', 'utils.py', 'aggregation.py', 'htr.py', 'evaluation.py', 'segmentation.py']
Imports OK


In [3]:
!pip install -q -r requirements.txt
print('Installation terminee.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 17.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 57.3 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.8/668.8 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 33.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.2/375.2 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 57.0 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 48.9 MB/s eta 0:00:0000:010

## 2 — Donnees CREMMA

In [4]:
import os, glob

data_dir = 'data/cremma-medieval'
if not os.path.exists(data_dir):
    !git clone https://github.com/HTR-United/cremma-medieval.git {data_dir}
else:
    print('Donnees deja presentes.')

xmls = [x for x in glob.glob('data/cremma-medieval/data/**/*.xml', recursive=True)
        if '.chocomufin' not in x]
print(f'{len(xmls)} fichiers XML ALTO')

Cloning into 'data/cremma-medieval'...
remote: Enumerating objects: 2249, done.
remote: Counting objects: 100% (854/854), done.
remote: Compressing objects: 100% (605/605), done.
remote: Total 2249 (delta 388), reused 679 (delta 241), pack-reused 1395 (from 1)
Receiving objects: 100% (2249/2249), 884.07 MiB | 29.15 MiB/s, done.
Resolving deltas: 100% (1225/1225), done.
Updating files: 100% (856/856), done.
279 fichiers XML ALTO


## 3 — Split + construction dataset lignes

In [5]:
import json
from pathlib import Path

split_path = Path('data/split.json')

if not split_path.exists():
    # Regenerer le split identique a celui produit en local (seed=42)
    data_dir = Path('data/cremma-medieval/data')
    manuscrits = sorted([str(d) for d in data_dir.iterdir() if d.is_dir()])

    VAL_SET  = {'bnf_arsenal_3516-imageDuMonde', 'bnf_fr_22549-septSages'}
    TEST_SET = {'bnf_fr_13496-saintJerome', 'kbr_9232-examensMoraux'}

    split = {
        'train': [m for m in manuscrits if Path(m).name not in VAL_SET | TEST_SET],
        'val'  : [m for m in manuscrits if Path(m).name in VAL_SET],
        'test' : [m for m in manuscrits if Path(m).name in TEST_SET],
    }
    split_path.write_text(json.dumps(split, indent=2, ensure_ascii=False))
    print('data/split.json cree.')
else:
    split = json.loads(split_path.read_text())
    print('data/split.json charge.')

for k, v in split.items():
    print(f'  {k}: {len(v)} manuscrits')

data/split.json cree.
  train: 10 manuscrits
  val: 2 manuscrits
  test: 2 manuscrits


In [6]:
from src.utils import fixer_seeds
from src.htr   import construire_dataset
from pathlib   import Path

fixer_seeds(42)

print('=== Construction du dataset lignes ===')
dataset = construire_dataset(
    split_path=Path('data/split.json'),
    out_dir=Path('data/lignes'),
    forcer=False,
)

print(f'\n{"Split":<8} {"Total":>7} {"Abrev.":>7} {"Taux":>7}')
print('-' * 30)
for spl, lignes in dataset.items():
    nb = sum(1 for l in lignes if l['has_abbreviation'])
    print(f'{spl:<8} {len(lignes):>7} {nb:>7} {nb/max(1,len(lignes)):>7.1%}')

=== Construction du dataset lignes ===
[info] train:   6891 lignes  (1678 avec abréviations, 24.4%)
[info] val  :   2733 lignes  (382 avec abréviations, 14.0%)
[info] test :    251 lignes  (47 avec abréviations, 18.7%)

Split      Total  Abrev.    Taux
------------------------------
train       6891    1678   24.4%
val         2733     382   14.0%
test         251      47   18.7%


## 4 — Baseline TrOCR zero-shot
CER avant tout fine-tuning — repond partiellement a la problematique.

In [7]:
from src.htr import evaluer_baseline_trocr
from pathlib  import Path

results_baseline = evaluer_baseline_trocr(
    manifest_path=Path('data/lignes/manifest_val.json'),
    max_lignes=200,
    batch_size=8,
)

print(f'CER global       : {results_baseline["CER_global"]:.4f}')
print(f'CER abreviations : {results_baseline["CER_abbrev"]}')
print(f'CER sans abrev.  : {results_baseline["CER_no_abbrev"]}')
print(f'delta_CER        : {results_baseline["delta_CER"]}  <-- impact abreviations')

[info] Inférence TrOCR baseline sur 200 lignes ...


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1610: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


CER global       : 0.9908
CER abreviations : 0.8343
CER sans abrev.  : 0.9948
delta_CER        : -0.1606  <-- impact abreviations


## 5 — Fine-tuning LoRA r=8
Early stopping patience=3 epochs sur CER val.

In [8]:
from src.htr import fine_tuner_trocr_lora
from pathlib  import Path

# Sur Kaggle les outputs vont dans /kaggle/working/
CKPT_DIR = Path('/kaggle/working/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

results_r8 = fine_tuner_trocr_lora(
    train_manifest=Path('data/lignes/manifest_train.json'),
    val_manifest  =Path('data/lignes/manifest_val.json'),
    lora_r=8, epochs=7, batch_size=16, learning_rate=5e-5, patience=3,
    max_val_lignes=500, seed=42,
    checkpoint_dir=CKPT_DIR,
)
print(f'r=8  -- CER val : {results_r8["meilleur_cer_val"]:.4f}  arret epoch {results_r8["epoch_arret"]}')


[info] Fine-tuning TrOCR LoRA r=8 sur cuda | fp16=True


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 3,846,144 || all params: 337,767,936 || trainable%: 1.1387


/kaggle/working/repo_htr/src/htr.py:541: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_fp16)
/kaggle/working/repo_htr/src/htr.py:556: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_fp16):
/kaggle/working/repo_htr/src/htr.py:575: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_fp16):
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1610: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


  Epoch  1 | loss=2.7759 | CER_val=0.3348 | CER_abbrev=0.3234 | CER_no_abbrev=0.3358
  Epoch  2 | loss=1.1504 | CER_val=0.2555 | CER_abbrev=0.2422 | CER_no_abbrev=0.2567
  Epoch  3 | loss=0.7786 | CER_val=0.2268 | CER_abbrev=0.2043 | CER_no_abbrev=0.2288
  Epoch  4 | loss=0.6089 | CER_val=0.2248 | CER_abbrev=0.2049 | CER_no_abbrev=0.2266
  Epoch  5 | loss=0.5043 | CER_val=0.2247 | CER_abbrev=0.1946 | CER_no_abbrev=0.2274
  Epoch  6 | loss=0.4343 | CER_val=0.2167 | CER_abbrev=0.1927 | CER_no_abbrev=0.2189
  Epoch  7 | loss=0.3742 | CER_val=0.2275 | CER_abbrev=0.2001 | CER_no_abbrev=0.23
r=8  -- CER val : 0.2167  arret epoch 7


## 5c — Sauvegarder le checkpoint (IMPORTANT)
Apres le fine-tuning, **faites Save Version** (bouton en haut a droite) pour que le checkpoint survive a la fin de session.

Ce notebook liste aussi le contenu de `/kaggle/working/` pour verification.

In [9]:
from pathlib import Path

ckpt_dir = Path('/kaggle/working/checkpoints')
ckpts = sorted(ckpt_dir.glob('trocr_lora_r*_best'))

if not ckpts:
    print('ATTENTION : aucun checkpoint dans', ckpt_dir)
else:
    for ck in ckpts:
        files = list(ck.rglob('*'))
        size_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
        print(f'Checkpoint : {ck.name}  ({size_mb:.0f} MB, {len(files)} fichiers)')
    print('Faites Save Version pour conserver ces fichiers dans Output Kaggle.')


Checkpoint : trocr_lora_r8_best  (19 MB, 6 fichiers)
Faites Save Version pour conserver ces fichiers dans Output Kaggle.


## 5b — Charger le checkpoint (modèle déjà entraîné)
À exécuter à la place de la section 5 si le fine-tuning a déjà été fait.

### Diagnostic — trouver le checkpoint Kaggle

In [10]:
import os
from pathlib import Path

print('=== /kaggle/working/checkpoints ===')
for p in sorted(Path('/kaggle/working').rglob('adapter_config.json')):
    print(' ', p.parent)

print('=== /kaggle/input (datasets attaches) ===')
for p in sorted(Path('/kaggle/input').rglob('adapter_config.json')):
    print(' ', p.parent)

print('=== Tout /kaggle/input ===')
for d in sorted(Path('/kaggle/input').iterdir()):
    print(' ', d)


=== /kaggle/working/checkpoints ===
  /kaggle/working/checkpoints/trocr_lora_r8_best
=== /kaggle/input (datasets attaches) ===
=== Tout /kaggle/input ===


In [11]:
from pathlib import Path

# Si tu as trouve le chemin avec la cellule de diagnostic, colle-le ici :
MANUAL_CKPT = ''  # ex: '/kaggle/input/mon-notebook/trocr_lora_r8_best'

if 'results_r8' in dir():
    best_ckpt = results_r8['checkpoint_path']
    print(f'Checkpoint session : {best_ckpt}')
elif MANUAL_CKPT:
    best_ckpt = MANUAL_CKPT
    print(f'Checkpoint manuel  : {best_ckpt}')
else:
    candidates = sorted(Path('/kaggle/working/checkpoints').glob('trocr_lora_r*_best'))
    if not candidates:
        candidates = sorted(Path('/kaggle/input').glob('*/trocr_lora_r*_best'))
    if not candidates:
        candidates = sorted(Path('/kaggle/input').rglob('adapter_config.json'))
        candidates = [p.parent for p in candidates]
    if not candidates:
        raise FileNotFoundError(
            'Checkpoint introuvable. Lance la cellule de diagnostic au-dessus, '
            'puis colle le chemin dans MANUAL_CKPT.'
        )
    best_ckpt = str(candidates[0])
    print(f'Checkpoint trouve  : {best_ckpt}')


Checkpoint session : /kaggle/working/checkpoints/trocr_lora_r8_best


## 6 — Evaluation finale (test set scelle)

In [12]:
import json
from pathlib        import Path
from PIL            import Image
from src.htr        import inferer_trocr
from src.evaluation import rapport_evaluation

test_items  = json.loads(Path('data/lignes/manifest_test.json').read_text())
test_images = [Image.open(it['image_path']).convert('RGB') for it in test_items]
test_refs   = [it['text'] for it in test_items]
test_flags  = [it['has_abbreviation'] for it in test_items]

# best_ckpt defini par la section 5 (lora-r8) ou 5b (load-checkpoint)
test_preds = inferer_trocr(test_images, model_path=best_ckpt, batch_size=8)
test_preds_baseline = inferer_trocr(test_images, batch_size=8)

rapport = rapport_evaluation(
    predictions=test_preds,
    references=test_refs,
    has_abbreviations=test_flags,
    out_path=Path('experiments/rapport_test_final.json'),
    predictions_modele2=test_preds_baseline,
    n_bootstrap=1000,
    seed=42,
)

cer = rapport['cer']
ic  = rapport['bootstrap_ic']
mcn = rapport['mcnemar']
print(f'CER global       : {cer["CER_global"]:.4f}  IC95% [{ic["CER_global"]["lower"]:.4f}, {ic["CER_global"]["upper"]:.4f}]')
print(f'CER abreviations : {cer["CER_abbrev"]}')
print(f'CER sans abrev.  : {cer["CER_no_abbrev"]}')
print(f'delta_CER        : {cer["delta_CER"]}  <-- reponse a la problematique')
print(f'WER global       : {rapport["wer"]["global"]:.4f}')
print(f'McNemar          : chi2={mcn["chi2"]}, p={mcn["p_value"]}, sig={mcn["significatif"]}')


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[info] Rapport écrit -> experiments/rapport_test_final.json

=== Rapport d'évaluation HTR ===
CER global      : 0.1418
CER abréviations: 0.1683
CER sans abrév. : 0.1357
delta_CER       : 0.0326
WER global      : 0.4926
IC 95% CER_global : [0.1260, 0.1592]

Conclusion :
Le CER global est 0.142. Sur les lignes contenant des abréviations médiévales, le CER est plus élevé (0.168) par rapport aux lignes ordinaires (0.136) (IC 95% bootstrap : [-0.005, 0.076]). Les caractères d'abréviation les plus impactants sont : «ͤ», «ꝯ», «ͬ». Les abréviations représentent 18.7% du corpus. Ces résultats indiquent que les abréviations impactent modérément les performances HTR sur le corpus CREMMA.

McNemar : chi2=18.05, p=0.0, significatif=OUI
CER global       : 0.1418  IC95% [0.1260, 0.1592]
CER abreviations : 0.1683
CER sans abrev.  : 0.1357
delta_CER        : 0.0326  <-- reponse a la problematique
WER global       : 0.4926
McNemar          : chi2=18.05, p=0.0, sig=True


## 6b — Paires prédiction / référence (inspection des erreurs)

In [13]:
import json, editdistance
from pathlib import Path

comparison = []
for item, pred in zip(test_items, test_preds):
    ref = item['text']
    cer = editdistance.eval(pred, ref) / max(1, len(ref))
    comparison.append({
        'line_id':          item['line_id'],
        'manuscrit':        item['manuscrit'],
        'reference':        ref,
        'prediction':       pred,
        'has_abbreviation': item['has_abbreviation'],
        'abreviations':     item.get('abreviations', []),
        'cer':              round(cer, 4),
    })

out = Path('experiments/predictions_test.json')
out.write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')

nb_ok  = sum(1 for c in comparison if c['cer'] == 0)
nb_err = len(comparison) - nb_ok
cer_moy = sum(c['cer'] for c in comparison) / max(1, len(comparison))
print(f'{len(comparison)} paires -> {out}')
print(f'  Parfaites (CER=0) : {nb_ok}')
print(f'  Avec erreurs      : {nb_err}')
print(f'  CER moyen         : {cer_moy:.4f}')

worst = sorted(comparison, key=lambda x: -x['cer'])[:5]
print('Top 5 erreurs :')
for w in worst:
    print(f'  CER={w["cer"]:.3f} | ref : {w["reference"][:60]}')
    print(f'           pred: {w["prediction"][:60]}')


251 paires -> experiments/predictions_test.json
  Parfaites (CER=0) : 20
  Avec erreurs      : 231
  CER moyen         : 0.1418
Top 5 erreurs :
  CER=1.000 | ref : E
           pred: 
  CER=1.000 | ref : Q
           pred: C
  CER=0.852 | ref : ci ꝯmcela vie .s'. jeroisme
           pred: t ͣg m͛t liv livie ·e ·e · �
  CER=0.500 | ref : nsint cõ
           pred: uisint c͛t
  CER=0.500 | ref : ge demora sainz jeroismes .iiii.
           pred: ge deniora sainz ꝯ ꝯrevilines ·


## 7 — dataset_nlp/ (JSON livrable)

In [ ]:
import json
from pathlib         import Path
from PIL             import Image
from src.htr         import inferer_trocr
from src.aggregation import generer_dataset_nlp

val_items  = json.loads(Path('data/lignes/manifest_val.json').read_text())
val_images = [Image.open(it['image_path']).convert('RGB') for it in val_items]
val_preds  = inferer_trocr(val_images, model_path=best_ckpt, batch_size=8)

dataset_val = generer_dataset_nlp(
    manifest_path=Path('data/lignes/manifest_val.json'),
    out_path=Path('dataset_nlp/val.json'),
    predictions=val_preds,
    confidences=[0.85] * len(val_preds),
)
dataset_test = generer_dataset_nlp(
    manifest_path=Path('data/lignes/manifest_test.json'),
    out_path=Path('dataset_nlp/test.json'),
    predictions=test_preds,
    confidences=[0.85] * len(test_preds),
)
generer_dataset_nlp(
    manifest_path=Path('data/lignes/manifest_val.json'),
    out_path=Path('dataset_nlp/val_oracle.json'),
)
print(f'val.json  : {len(dataset_val)} entrees')
print(f'test.json : {len(dataset_test)} entrees')

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 7b — Générer dataset_nlp/train.json (manquant)
`train.json` n'était pas produit — on l'ajoute ici avec les prédictions du modèle fine-tuné.

In [ ]:
import json
from pathlib import Path
from PIL     import Image
from src.htr         import inferer_trocr
from src.aggregation import generer_dataset_nlp

train_items  = json.loads(Path('data/lignes/manifest_train.json').read_text())
train_images = [Image.open(it['image_path']).convert('RGB') for it in train_items]
print(f'Inférence sur {len(train_images)} lignes train (~10 min sur T4)...')
train_preds = inferer_trocr(train_images, model_path=best_ckpt, batch_size=8)

dataset_train = generer_dataset_nlp(
    manifest_path=Path('data/lignes/manifest_train.json'),
    out_path=Path('dataset_nlp/train.json'),
    predictions=train_preds,
    confidences=[0.85] * len(train_preds),
)
print(f'train.json : {len(dataset_train)} entrées')


## 8 — MODEL_CARD.md

In [ ]:
from pathlib import Path

cer_g  = rapport['cer']['CER_global']
cer_a  = rapport['cer']['CER_abbrev']
cer_n  = rapport['cer']['CER_no_abbrev']
wer_g  = rapport['wer']['global']
ic_g   = rapport['bootstrap_ic']['CER_global']
delta  = rapport['cer']['delta_CER']
nb_rev = sum(1 for e in dataset_test if e['needs_review'])
taux_r = nb_rev / max(1, len(dataset_test))

card = f"""# Model Card - HTR CREMMA Medieval 2026

## Description
Modele HTR fine-tune sur CREMMA Medieval (CC-BY 4.0). Vieux/moyen francais XIe-XVIIe s.

## Donnees
| Split | Manuscrits | Lignes | SHA-256 |
|---|---|---|---|
| Train | 10 | 6891 | de26aaa226edad415b37e02a6e3e68a5f001968a3ec69daddb3c47b69d9767d8 |
| Val   | 2  | 2733 | b7891b9a73b9704e37623b7dfec43677a7a94da2f58aa94557254dfd1a03a4c5 |
| Test  | 2  |  251 | 86f5f48b7628128b4e86bc5684b13f7409972bf3d7dc39291d97b03d25f62949 |

## Architecture
| Composant | Detail |
|---|---|
| Modele base | microsoft/trocr-base-handwritten |
| Fine-tuning | LoRA r=8, alpha=32, dropout=0.1 |
| Optimiseur  | AdamW lr=5e-5, patience=3, fp16, gradient_checkpointing |

## Performances (test set scelle)
| Metrique | Valeur | IC 95% |
|---|---|---|
| CER global | {cer_g:.4f} ({cer_g:.1%}) | [{ic_g['lower']:.4f}, {ic_g['upper']:.4f}] |
| CER abrev. | {cer_a} | - |
| CER normal | {cer_n} | - |
| delta_CER  | {delta} | - |
| WER global | {wer_g:.4f} ({wer_g:.1%}) | - |
| needs_review | {taux_r:.1%} | - |

## Conclusion
{rapport['conclusion_problematique']}
"""

Path('MODEL_CARD.md').write_text(card, encoding='utf-8')
print('MODEL_CARD.md mis a jour.')

## 9 — Telechargement des artefacts

In [ ]:
from pathlib import Path

# Sur Kaggle, les fichiers /kaggle/working/ sont dans Output apres Save Version.
artefacts = [
    'experiments/journal.jsonl',
    'experiments/rapport_test_final.json',
    'experiments/predictions_test.json',
    'MODEL_CARD.md',
    'dataset_nlp/val.json',
    'dataset_nlp/test.json',
    'dataset_nlp/val_oracle.json',
    'dataset_nlp/train.json',
]

repo = Path('/kaggle/working/repo_htr')
print('=== Artefacts ===')
for p in artefacts:
    full = repo / p
    if full.exists():
        print(f'  OK  {full.stat().st_size / 1024:8.1f} KB  {p}')
    else:
        print(f'  --  ABSENT              {p}')

print('=== Checkpoints ===')
for ck in sorted(Path('/kaggle/working/checkpoints').glob('trocr_lora_r*_best')):
    size_mb = sum(f.stat().st_size for f in ck.rglob('*') if f.is_file()) / 1e6
    print(f'  {size_mb:6.0f} MB  {ck}')

print('Pour telecharger : Save Version > Output > Download All')
